# Phase 5.0: Setup and Sanity Check

This script does the plumbing every later Phase 5 model will need:
1. Loads Phase 2 segments and labels (binarized: Relaxed vs Stress).
2. Splits 40 subjects into 32 train / 4 val / 4 test (no leakage).
3. Builds PyTorch DataLoaders.
4. Sanity-checks: pulls one batch, prints shapes, checks class balance.
5. Saves the split assignment to Results/phase5/ for reproducibility.
6. Plots class balance per split.

Run this ONCE before phase5_1, phase5_2, etc. You don't need to re-run it
every time, `phase5_utils.make_subject_split(seed=SEED)` is deterministic,
so any later script that calls it gets the same split.

In [1]:
import os
import json
import numpy as np
import matplotlib.pyplot as plt
import torch
 
from phase5_utils import (
    load_phase2_data,
    make_subject_split,
    make_loaders,
    PHASE5_DIR,
    SEED,
)

os.makedirs(PHASE5_DIR, exist_ok=True)

# Reproducibility. seed numpy and torch up front
np.random.seed(SEED)
torch.manual_seed(SEED)

# Load Data

In [2]:
print("\n[1/5] Loading Phase 2 segments and labels...")
X, y_binary, subjects = load_phase2_data(verbose=True)


[1/5] Loading Phase 2 segments and labels...
  NPZ keys: ['segments']
  CSV columns: ['subject', 'task', 'trial', 'window', 'rating', 'stress_level']
  CSV head:
 subject       task  trial  window  rating stress_level
       1 Relaxation      1       1       0      Relaxed
       1 Relaxation      1       2       0      Relaxed
       1 Relaxation      1       3       0      Relaxed
       1 Relaxation      1       4       0      Relaxed
       1 Relaxation      1       5       0      Relaxed

  X shape: (2400, 32, 640)  dtype: float32
  y_binary classes: {0: 1110, 1: 1290}
  unique subjects: 40


# Subject-grouped Split (32 / 4 / 4)

In [3]:
print("\n[2/5] Splitting 40 subjects into 32 train / 4 val / 4 test...")

train_idx, val_idx, test_idx, split_info = make_subject_split(subjects, n_train=32, n_val=4, n_test=4, seed=SEED)
 
print(f"  Train subjects ({len(split_info['train'])}): {split_info['train']}")
print(f"  Val   subjects ({len(split_info['val'])}):  {split_info['val']}")
print(f"  Test  subjects ({len(split_info['test'])}): {split_info['test']}")
 
print(f"\n  Train segments: {train_idx.sum()}")
print(f"  Val   segments: {val_idx.sum()}")
print(f"  Test  segments: {test_idx.sum()}")
print(f"  Total          : {train_idx.sum() + val_idx.sum() + test_idx.sum()} "
      f"(expected {len(subjects)})")


[2/5] Splitting 40 subjects into 32 train / 4 val / 4 test...
  Train subjects (32): [1, 4, 5, 6, 7, 8, 10, 11, 12, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 35, 36, 37, 38, 40]
  Val   subjects (4):  [3, 13, 15, 39]
  Test  subjects (4): [2, 9, 14, 34]

  Train segments: 1920
  Val   segments: 240
  Test  segments: 240
  Total          : 2400 (expected 2400)


# Class balance per split

make sure no split is pathologically skewed

In [4]:
print("\n[3/5] Class balance per split (binary: 0=Relaxed, 1=Stress):")
print("  " + "-" * 56)
print(f"  {'Split':6s} | {'Relaxed':>8s} | {'Stress':>8s} | {'%Stress':>8s}")
print("  " + "-" * 56)
for name, idx in [("Train", train_idx), ("Val", val_idx), ("Test", test_idx)]:
    y_split = y_binary[idx]
    n0 = int((y_split == 0).sum())
    n1 = int((y_split == 1).sum())
    pct = n1 / max(len(y_split), 1) * 100
    print(f"  {name:6s} | {n0:8d} | {n1:8d} | {pct:7.1f}%")
print("  " + "-" * 56)


[3/5] Class balance per split (binary: 0=Relaxed, 1=Stress):
  --------------------------------------------------------
  Split  |  Relaxed |   Stress |  %Stress
  --------------------------------------------------------
  Train  |      845 |     1075 |    56.0%
  Val    |      120 |      120 |    50.0%
  Test   |      145 |       95 |    39.6%
  --------------------------------------------------------


# Build DataLoaders and pull one batch to verify shapes

In [5]:
print("\n[4/5] Building DataLoaders and pulling one batch from train_loader...")

train_loader, val_loader, test_loader = make_loaders(X, y_binary, train_idx, val_idx, test_idx, batch_size=64)
 
X_batch, y_batch = next(iter(train_loader))
print(f"  Batch X shape: {tuple(X_batch.shape)}  dtype: {X_batch.dtype}")
print(f"    -> (batch_size, 1 image-channel, 32 EEG channels, 640 samples)")
print(f"  Batch y shape: {tuple(y_batch.shape)}  dtype: {y_batch.dtype}")
print(f"  X stats: mean={X_batch.mean().item():+.3f}, "
      f"std={X_batch.std().item():.3f}, "
      f"min={X_batch.min().item():+.2f}, "
      f"max={X_batch.max().item():+.2f}")
print(f"  -> Should be near-zero mean, ~1 std (per-subject z-scored in Phase 2).")


[4/5] Building DataLoaders and pulling one batch from train_loader...
  Batch X shape: (64, 1, 32, 640)  dtype: torch.float32
    -> (batch_size, 1 image-channel, 32 EEG channels, 640 samples)
  Batch y shape: (64,)  dtype: torch.int64
  X stats: mean=-0.002, std=0.989, min=-4.16, max=+4.53
  -> Should be near-zero mean, ~1 std (per-subject z-scored in Phase 2).


# Saving split assignment for reproducibility

In [6]:
print("\n[5/5] Saving split assignment to Results/phase5/...")
split_path = os.path.join(PHASE5_DIR, "subject_split.json")
with open(split_path, "w") as f:
    json.dump(split_info, f, indent=2)
print(f"  Saved: {split_path}")


[5/5] Saving split assignment to Results/phase5/...
  Saved: C:\Users\hibro\OneDrive\Desktop\Desktop_Files\Projects\Python\ML_Models\Cognitive_Stress_Classification\EEG-Stress-Classification\Results\phase5\subject_split.json


# Visualization

class balance bar chart

In [7]:
fig, ax = plt.subplots(1, 1, figsize=(8, 4.5))
splits = ["Train (32 subj)", "Val (4 subj)", "Test (4 subj)"]
relaxed = [
    int((y_binary[train_idx] == 0).sum()),
    int((y_binary[val_idx]   == 0).sum()),
    int((y_binary[test_idx]  == 0).sum()),
]
stress = [
    int((y_binary[train_idx] == 1).sum()),
    int((y_binary[val_idx]   == 1).sum()),
    int((y_binary[test_idx]  == 1).sum()),
]
x = np.arange(len(splits))
w = 0.35
bars0 = ax.bar(x - w/2, relaxed, w, label="Relaxed (0)", color="#5DA5DA")
bars1 = ax.bar(x + w/2, stress,  w, label="Stress (1)",  color="#F15854")
ax.set_xticks(x)
ax.set_xticklabels(splits)
ax.set_ylabel("Number of segments")
ax.set_title("Phase 5 — Class balance per split (binary classification)")
ax.legend()
ax.grid(axis="y", alpha=0.3)
 
# Numeric labels on top of each bar
for bar, v in list(zip(bars0, relaxed)) + list(zip(bars1, stress)):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 5,
            str(v), ha="center", fontsize=9)
 
plt.tight_layout()
fig_path = os.path.join(PHASE5_DIR, "class_balance_per_split.png")
plt.savefig(fig_path, dpi=150)
plt.close()
print(f"  Saved class balance plot: {fig_path}")
 
 
print("\n" + "=" * 70)
print("Phase 5.0 setup complete.")
print("=" * 70)

  Saved class balance plot: C:\Users\hibro\OneDrive\Desktop\Desktop_Files\Projects\Python\ML_Models\Cognitive_Stress_Classification\EEG-Stress-Classification\Results\phase5\class_balance_per_split.png

Phase 5.0 setup complete.
